<div style="background:linear-gradient(135deg,#0D0F1A 0%,#151828 60%,#0D0F1A 100%);border:1px solid #1E2340;border-radius:14px;padding:36px 40px;font-family:'Segoe UI',sans-serif;">
<h1 style="color:#00C8FF;font-size:2.4em;margin:0 0 6px;">⚡ Cobalt AI</h1>
<h2 style="color:#7B2FFF;font-size:1.3em;font-weight:400;margin:0 0 18px;">Transformer Model — From Scratch in Rust</h2>
<p style="color:#9AADCC;font-size:0.95em;line-height:1.7;max-width:680px;">
This notebook is your end-to-end guide to the <strong style='color:#E0E6FF'>CobaltTransformer</strong> — a GPT-style
autoregressive language model built entirely in <strong style='color:#E0E6FF'>Rust</strong> using the
<code style='color:#00C8FF'>burn-rs</code> deep learning framework.<br><br>
We cover: environment setup → model config inspection → parameter breakdown → 
training metrics visualisation → live Rust inference → batch generation.
</p>
<hr style="border-color:#1E2340;margin:20px 0;"/>
<table style="color:#9AADCC;font-size:0.88em;border-collapse:collapse;">
<tr><td style="padding:3px 16px 3px 0;">🦀 <strong>Backend</strong></td><td>Rust + burn 0.20 + WGPU</td></tr>
<tr><td style="padding:3px 16px 3px 0;">🧠 <strong>Architecture</strong></td><td>3-layer Transformer, d=192, 4 heads</td></tr>
<tr><td style="padding:3px 16px 3px 0;">📦 <strong>Model file</strong></td><td>models/cobalt_model.mpk (MessagePack)</td></tr>
<tr><td style="padding:3px 16px 3px 0;">📓 <strong>Version</strong></td><td>Cobalt AI v1.1.0</td></tr>
</table>
</div>

## 0 · Environment Setup

In [ ]:
import sys
import json
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ── Add notebooks dir to path so we can import cobalt_utils ────────────────
sys.path.insert(0, str(Path.cwd()))
import cobalt_utils as cu

# ── Sanity check ───────────────────────────────────────────────────────────
print(f"✅  Python   : {sys.version.split()[0]}")
print(f"📂  Root     : {cu.ROOT_DIR}")
print(f"📦  Model    : {cu.MODEL_PATH}  ({'exists ✅' if cu.MODEL_PATH.exists() else '⚠ not found — run: cargo run -- train'})")
print(f"🦀  Binary   : {cu.BINARY_PATH} ({'exists ✅' if cu.BINARY_PATH.exists() else '⚠ not built  — run: cargo build --release'})")

try:
    import torch
    print(f"🔥  PyTorch  : {torch.__version__}")
except ImportError:
    print("⚠  PyTorch not installed — pip install torch")

print("\n🚀  Cobalt AI notebook ready!")

---
## 1 · Model Configuration
Load `models/config.json` — the authoritative source of truth for this model's architecture.

In [ ]:
cfg = cu.load_config()

if cfg:
    cu.print_config(cfg)
else:
    print("⚠  Could not load config.json")

### 1.1 · Parameter Count Breakdown
Estimate total trainable parameters directly from the config — no model loading required.

In [ ]:
if cfg:
    cu.print_param_breakdown(cfg)

    # Visual bar chart
    cu.apply_cobalt_theme()
    breakdown = cu.count_parameters(cfg)
    breakdown.pop("TOTAL", None)

    labels = [k.replace(" ", "\n") for k in breakdown]
    values = list(breakdown.values())
    colours = [cu.COBALT_BLUE, cu.COBALT_CYAN, cu.COBALT_PURPLE, cu.COBALT_BLUE]

    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.barh(labels, values, color=colours[:len(values)], height=0.5)

    for bar, val in zip(bars, values):
        ax.text(bar.get_width() + max(values)*0.01, bar.get_y() + bar.get_height()/2,
                f"{val:,}", va="center", color=cu.COBALT_TEXT, fontsize=9)

    ax.set_xlabel("Parameters")
    ax.set_title("CobaltTransformer — Parameter Distribution", fontsize=15, pad=12)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x/1e3:.0f}k"))
    plt.tight_layout()
    plt.show()

---
## 2 · Training Metrics
Visualise the loss curve from `experiments/training_logs.json`.

In [ ]:
logs = cu.load_training_logs()

if logs:
    print(f"📊  Loaded {len(logs)} log entries")
    print(f"    First loss : {logs[0]['loss']:.4f}")
    print(f"    Final loss : {logs[-1]['loss']:.4f}")
    print(f"    Reduction  : {(1 - logs[-1]['loss']/logs[0]['loss'])*100:.1f}%")
    cu.plot_loss_curve(logs, smooth=True)
else:
    print("⚠  No training logs found.")

### 2.1 · Per-Epoch Summary

In [ ]:
if logs:
    from collections import defaultdict

    epoch_losses = defaultdict(list)
    for entry in logs:
        epoch_losses[entry["epoch"]].append(entry["loss"])

    epochs   = sorted(epoch_losses)
    avg_loss = [np.mean(epoch_losses[e]) for e in epochs]
    min_loss = [np.min(epoch_losses[e])  for e in epochs]

    cu.apply_cobalt_theme()
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(epochs, avg_loss, color=cu.COBALT_BLUE,   marker="o", label="Avg Loss / Epoch")
    ax.plot(epochs, min_loss, color=cu.COBALT_PURPLE, marker="s", linestyle="--", label="Min Loss / Epoch")
    ax.fill_between(epochs, min_loss, avg_loss, color=cu.COBALT_BLUE, alpha=0.08)

    ax.set_title("Training Progress — Per-Epoch Summary", fontsize=15, pad=12)
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_xticks(epochs)
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Print table
    print(f"{'Epoch':>6}  {'Avg Loss':>10}  {'Min Loss':>10}")
    print("─" * 32)
    for e, avg, mn in zip(epochs, avg_loss, min_loss):
        print(f"{e:>6}  {avg:>10.4f}  {mn:>10.4f}")

---
## 3 · Rust Inference
Call the compiled Rust binary directly from Python using `subprocess`.

In [ ]:
# ── Single generation ───────────────────────────────────────────────────────
prompt = "ROMEO:"
output = cu.run_rust_inference(prompt, max_tokens=200)

### 3.1 · Batch Generation — Multiple Prompts

In [ ]:
prompts = [
    "HAMLET:",
    "To be, or not to be",
    "JULIET:",
    "All the world's a stage",
]

print("🚀 Batch Generations\n" + "═" * 60)
for p in prompts:
    print(f"\n📝  Prompt: {p}")
    print("─" * 60)
    cu.run_rust_inference(p, max_tokens=80)
    print()

---
## 4 · Architecture Deep Dive
Understand the model structure visually.

In [ ]:
cu.apply_cobalt_theme()

if cfg:
    arch  = cfg["architecture"]
    d     = arch["d_model"]
    n_h   = arch["n_heads"]
    n_l   = arch["n_layers"]
    d_ff  = arch["d_ff"]
    head_dim = d // n_h

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # ── Left: attention head dimension breakdown ────────────────────────────
    ax = axes[0]
    labels  = [f"Head {i+1}\n({head_dim}d)" for i in range(n_h)]
    sizes   = [head_dim] * n_h
    colours = [cu.COBALT_BLUE, cu.COBALT_CYAN, cu.COBALT_PURPLE, "#FF6B6B"][:n_h]
    wedges, texts, autotexts = ax.pie(
        sizes, labels=labels, colors=colours,
        autopct="%1.0f%%", startangle=90,
        wedgeprops={"linewidth": 2, "edgecolor": cu.COBALT_BG}
    )
    for t in autotexts: t.set_color(cu.COBALT_TEXT)
    ax.set_title(f"Multi-Head Attention\n{n_h} heads × {head_dim}d = {d}d", pad=14)

    # ── Right: layer dimension flow ─────────────────────────────────────────
    ax = axes[1]
    components = ["Token\nEmbed", "Pos\nEmbed", "Attn\n(per block)", "FFN\n(per block)", "Output\nHead"]
    dims       = [d, d, d, d_ff, d]
    bar_colours = [cu.COBALT_BLUE, cu.COBALT_CYAN, cu.COBALT_PURPLE, cu.COBALT_PURPLE, cu.COBALT_BLUE]
    bars = ax.bar(components, dims, color=bar_colours, width=0.55)
    for bar, val in zip(bars, dims):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 12,
                str(val), ha="center", color=cu.COBALT_TEXT, fontsize=10, fontweight="bold")
    ax.set_ylabel("Dimension")
    ax.set_title(f"Dimension Flow ({n_l} Transformer Blocks)", pad=14)
    ax.set_ylim(0, d_ff * 1.15)

    plt.suptitle("CobaltTransformer — Architecture Overview", fontsize=16, y=1.02, color=cu.COBALT_CYAN)
    plt.tight_layout()
    plt.show()

---
## 5 · Retrain from Scratch
Kick off a fresh training run directly from this notebook.

In [ ]:
# ⚠  This will actually run training (14 epochs × 250 iters).
# Comment out if you just want to explore the notebook.

RETRAIN = False   # ← flip to True to start training

if RETRAIN:
    manifest = cu.ROOT_DIR / "cobalt_ai" / "Cargo.toml"
    cmd = ["cargo", "run", f"--manifest-path={manifest}", "--release", "--", "train"]
    print(f"🦀  Running: {' '.join(cmd)}\n")
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                             text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    print(f"\n{'✅ Training complete!' if proc.returncode == 0 else '❌ Training failed.'}")
else:
    print("ℹ  RETRAIN = False — set it to True to kick off training.")

---
<div style="background:#0D0F1A;border:1px solid #1E2340;border-radius:10px;padding:20px 28px;font-family:'Segoe UI',sans-serif;">
<p style="color:#9AADCC;margin:0;">📌 <strong style='color:#00C8FF'>Next Steps</strong></p>
<ul style="color:#9AADCC;margin-top:10px;line-height:2;">
<li>Run <code>training_metrics.ipynb</code> for deeper metric analysis</li>
<li>Run <code>model_architecture.ipynb</code> for transformer block visualisation</li>
<li>Add a PyO3/Maturin binding to call Rust inference natively (no subprocess)</li>
<li>Swap the dataset in <code>data/input.txt</code> and retrain on custom text</li>
</ul>
</div>